# BÁO CÁO THỰC NGHIỆM: SO SÁNH 3 PHƯƠNG PHÁP FACE RECOGNITION TỪ SLIDE

Được thiết kế dựa trên các nội dung trong slide bài giảng:

| STT | Phương pháp | Cơ sở lý thuyết trong Slide | Đặc điểm huấn luyện |
| :---: | :--- | :--- | :--- |
| **1** | **Triplet Loss (Random Triplets)** | Slide 15-20: $L = \max(d(A,P) - d(A,N) + \alpha, 0)$ | Chọn ngẫu nhiên bộ ba $(A, P, N)$. Nhiều triplet quá dễ khiến loss = 0. |
| **2** | **Triplet Loss + Hard Triplet Mining** | Slide 19, 21: $d(A,P) \approx d(A,N)$ | Lọc online trong batch: chọn $P$ xa nhất và $N$ gần nhất để ép mạng hội tụ. |
| **3** | **Siamese Binary Classification** | Slide 24-28: $\hat{y} = \sigma(\sum w_k \|f(x_1)_k - f(x_2)_k\| + b)$ | Coi nhận diện là bài toán phân loại nhị phân trên vector hiệu số tuyệt đối. |

**Quy chuẩn thực nghiệm:** Cả 3 phương pháp cùng chạy trên:
- **Cùng Dataset:** LFW (Labeled Faces in the Wild)
- **Cùng Backbone:** ResNet-18 trích xuất vector 128 chiều, chuẩn hóa $L_2$ ($||f(x)||_2 = 1$)
- **Cùng Optimizer:** Adam ($lr = 10^{-4}$)
- **Cùng số Epoch:** 8 epochs


## BƯỚC 1: KIỂM TRA THIẾT BỊ VÀ THƯ VIỆN


In [ ]:
import os, random, time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from sklearn.datasets import fetch_lfw_people
from sklearn.metrics import roc_curve, auc, accuracy_score
from sklearn.manifold import TSNE

# Cố định random seed để kết quả thực nghiệm có thể tái lập
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Thiết bị đang sử dụng: {device}")
if torch.cuda.is_available():
    print(f"[*] Tên GPU: {torch.cuda.get_device_name(0)}")


## BƯỚC 2: TẢI DATASET LFW (LABELED FACES IN THE WILD)


In [ ]:
print("[*] Đang tải tập dữ liệu LFW...")
lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.8, color=True)

images = lfw.images   # Shape: [N, H, W, 3], giá trị float 0.0 - 1.0
labels = lfw.target
target_names = lfw.target_names

label_to_indices = defaultdict(list)
for idx, lbl in enumerate(labels):
    label_to_indices[lbl].append(idx)

eligible_labels = [l for l, idxs in label_to_indices.items() if len(idxs) >= 2]
all_labels = list(label_to_indices.keys())

print(f"[*] Tổng số ảnh: {len(images)}")
print(f"[*] Tổng số cá nhân: {len(target_names)}")
print(f"[*] Số người có >= 2 ảnh: {len(eligible_labels)}")

# Hiển thị một số khuôn mặt mẫu
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(target_names[labels[i]].split()[-1], fontsize=9)
    ax.axis('off')
plt.suptitle("Các khuôn mặt mẫu trong tập dữ liệu LFW", fontsize=13)
plt.tight_layout()
plt.show()


## BƯỚC 3: ĐỊNH NGHĨA KIẾN TRÚC MẠNG & 3 HÀM LOSS


In [ ]:
# 1. Backbone chung: ResNet-18 -> 128-D vector, chuẩn hóa L2
class FaceEmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.head(feat)
        return F.normalize(emb, p=2, dim=1)  # ||f(x)||_2 = 1


# 2. Phương pháp 3: Binary Classification (Slide 24-28)
class SiameseBinaryNet(nn.Module):
    def __init__(self, embedding_net):
        super().__init__()
        self.embedding_net = embedding_net
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1)
        )

    def forward(self, x1, x2):
        f1 = self.embedding_net(x1)
        f2 = self.embedding_net(x2)
        diff = torch.abs(f1 - f2)
        logits = self.classifier(diff)
        return logits, f1, f2


# 3. Phương pháp 1: Triplet Loss cơ bản (Random Triplets) (Slide 16-17)
class TripletLoss(nn.Module):
    def __init__(self, alpha=0.2):
        super().__init__()
        self.alpha = alpha

    def forward(self, anchor, positive, negative):
        d_ap = F.pairwise_distance(anchor, positive, p=2).pow(2)
        d_an = F.pairwise_distance(anchor, negative, p=2).pow(2)
        loss = F.relu(d_ap - d_an + self.alpha)
        return loss.mean(), d_ap.mean().item(), d_an.mean().item()


# 4. Phương pháp 2: Batch Hard Triplet Loss (Slide 19, 21)
class BatchHardTripletLoss(nn.Module):
    def __init__(self, alpha=0.2):
        super().__init__()
        self.alpha = alpha

    def forward(self, embeddings, labels):
        # Ma trận bình phương khoảng cách: ||x-y||^2 = 2 - 2<x,y> (do đã L2-normalized)
        dot = torch.matmul(embeddings, embeddings.t())
        pdist = torch.clamp(2.0 - 2.0 * dot, min=0.0)

        labels = labels.unsqueeze(1)
        same_mask = torch.eq(labels, labels.t())

        # Hardest Positive: max d(A, P)
        hardest_pos, _ = torch.max(pdist * same_mask.float(), dim=1)

        # Hardest Negative: min d(A, N)
        max_d = torch.max(pdist)
        hardest_neg, _ = torch.min(pdist + max_d * same_mask.float(), dim=1)

        loss = F.relu(hardest_pos - hardest_neg + self.alpha)
        valid = loss[loss > 1e-16]
        if len(valid) == 0:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True), hardest_pos.mean().item(), hardest_neg.mean().item()
        return valid.mean(), hardest_pos.mean().item(), hardest_neg.mean().item()

print("[*] Đã khởi tạo đầy đủ kiến trúc mô hình và các hàm Loss!")


## BƯỚC 4: TIỀN XỬ LÝ & TẠO DATASET PHÙ HỢP TỪNG PHƯƠNG PHÁP


In [ ]:
img_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def to_tensor(idx):
    return img_transform((images[idx] * 255).astype(np.uint8))


# Dataset cho Triplet Loss (Random)
class TripletDataset(Dataset):
    def __init__(self, num_triplets=1500):
        self.num_triplets = num_triplets

    def __len__(self):
        return self.num_triplets

    def __getitem__(self, idx):
        lbl_p = random.choice(eligible_labels)
        idx_a, idx_p = random.sample(label_to_indices[lbl_p], 2)
        lbl_n = random.choice([l for l in all_labels if l != lbl_p])
        idx_n = random.choice(label_to_indices[lbl_n])
        return to_tensor(idx_a), to_tensor(idx_p), to_tensor(idx_n)


# Dataset cho Batch Hard Triplet Mining (mỗi mẫu: ảnh + person_id)
class FaceIDDataset(Dataset):
    def __init__(self):
        self.samples = []
        for lbl in eligible_labels:
            for idx in label_to_indices[lbl]:
                self.samples.append((idx, lbl))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        idx, lbl = self.samples[i]
        return to_tensor(idx), lbl


# Dataset cho Binary Classification (cặp ảnh + nhãn 0/1)
class PairDataset(Dataset):
    def __init__(self, num_pairs=1500):
        self.num_pairs = num_pairs

    def __len__(self):
        return self.num_pairs

    def __getitem__(self, idx):
        is_same = (idx % 2 == 0)
        if is_same:
            lbl = random.choice(eligible_labels)
            i1, i2 = random.sample(label_to_indices[lbl], 2)
            y = 1.0
        else:
            l1, l2 = random.sample(all_labels, 2)
            i1 = random.choice(label_to_indices[l1])
            i2 = random.choice(label_to_indices[l2])
            y = 0.0
        return to_tensor(i1), to_tensor(i2), torch.tensor(y, dtype=torch.float32)

triplet_loader = DataLoader(TripletDataset(1500), batch_size=32, shuffle=True)
hard_loader = DataLoader(FaceIDDataset(), batch_size=64, shuffle=True)
pair_loader = DataLoader(PairDataset(1500), batch_size=32, shuffle=True)

print(f"[*] Triplet Loader: {len(triplet_loader.dataset)} bộ ba")
print(f"[*] Hard Mining Loader: {len(hard_loader.dataset)} ảnh (Batch size = 64 để tìm Hard Negative)")
print(f"[*] Pair Loader: {len(pair_loader.dataset)} cặp ảnh")


## BƯỚC 5: HUẤN LUYỆN PHƯƠNG PHÁP 1 - TRIPLET LOSS (RANDOM TRIPLETS)
Bám sát Slide 16-20: Chọn ngẫu nhiên bộ ba $(A, P, N)$.


In [ ]:
NUM_EPOCHS = 8
LR = 1e-4
ALPHA = 0.2

print("=" * 60)
print("PHƯƠNG PHÁP 1: TRIPLET LOSS (Random Triplets) - Slide 16-20")
print("=" * 60)

model_triplet = FaceEmbeddingNet().to(device)
criterion_triplet = TripletLoss(alpha=ALPHA)
optimizer_triplet = torch.optim.Adam(model_triplet.parameters(), lr=LR)

history_triplet = {'loss': [], 'd_ap': [], 'd_an': []}

for epoch in range(1, NUM_EPOCHS + 1):
    model_triplet.train()
    total_loss, total_ap, total_an, count = 0.0, 0.0, 0.0, 0
    for a, p, n in triplet_loader:
        a, p, n = a.to(device), p.to(device), n.to(device)
        optimizer_triplet.zero_grad()
        fa, fp, fn = model_triplet(a), model_triplet(p), model_triplet(n)
        loss, ap, an = criterion_triplet(fa, fp, fn)
        loss.backward()
        optimizer_triplet.step()

        bs = a.size(0)
        total_loss += loss.item() * bs
        total_ap += ap * bs
        total_an += an * bs
        count += bs

    el = total_loss / count
    ea = total_ap / count
    en = total_an / count
    history_triplet['loss'].append(el)
    history_triplet['d_ap'].append(ea)
    history_triplet['d_an'].append(en)
    print(f"  Epoch [{epoch}/{NUM_EPOCHS}] | Loss: {el:.4f} | d(A,P): {ea:.3f} | d(A,N): {en:.3f}")

print("[✓] Phương pháp 1 huấn luyện xong!\n")


## BƯỚC 6: HUẤN LUYỆN PHƯƠNG PHÁP 2 - TRIPLET LOSS (HARD MINING)
Bám sát Slide 19, 21: *"Choose triplets that're hard to train on: $d(A,P) + \alpha \le d(A,N)$ and $d(A,P) \approx d(A,N)$"*.


In [ ]:
print("=" * 60)
print("PHƯƠNG PHÁP 2: TRIPLET LOSS + HARD MINING - Slide 19, 21")
print("=" * 60)

model_hard = FaceEmbeddingNet().to(device)
criterion_hard = BatchHardTripletLoss(alpha=ALPHA)
optimizer_hard = torch.optim.Adam(model_hard.parameters(), lr=LR)

history_hard = {'loss': [], 'd_ap': [], 'd_an': []}

for epoch in range(1, NUM_EPOCHS + 1):
    model_hard.train()
    total_loss, total_ap, total_an, count = 0.0, 0.0, 0.0, 0
    for imgs, lbls in hard_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        if len(torch.unique(lbls)) < 2:
            continue
        optimizer_hard.zero_grad()
        embeddings = model_hard(imgs)
        loss, ap, an = criterion_hard(embeddings, lbls)
        loss.backward()
        optimizer_hard.step()

        bs = imgs.size(0)
        total_loss += loss.item() * bs
        total_ap += ap * bs
        total_an += an * bs
        count += bs

    el = total_loss / count if count > 0 else 0
    ea = total_ap / count if count > 0 else 0
    en = total_an / count if count > 0 else 0
    history_hard['loss'].append(el)
    history_hard['d_ap'].append(ea)
    history_hard['d_an'].append(en)
    print(f"  Epoch [{epoch}/{NUM_EPOCHS}] | Loss: {el:.4f} | d(A,P): {ea:.3f} | d(A,N): {en:.3f}")

print("[✓] Phương pháp 2 huấn luyện xong!\n")


## BƯỚC 7: HUẤN LUYỆN PHƯƠNG PHÁP 3 - SIAMESE BINARY CLASSIFICATION
Bám sát Slide 24-28: Dự đoán nhị phân $\hat{y} = \sigma(W \cdot |f(x_1) - f(x_2)| + b)$ qua BCE Loss.


In [ ]:
print("=" * 60)
print("PHƯƠNG PHÁP 3: BINARY CLASSIFICATION - Slide 24-28")
print("=" * 60)

binary_emb_net = FaceEmbeddingNet().to(device)
model_binary = SiameseBinaryNet(binary_emb_net).to(device)
criterion_bce = nn.BCEWithLogitsLoss()
optimizer_bce = torch.optim.Adam(model_binary.parameters(), lr=LR)

history_binary = {'loss': [], 'acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    model_binary.train()
    total_loss, correct, total = 0.0, 0, 0
    for x1, x2, y in pair_loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        optimizer_bce.zero_grad()
        logits, _, _ = model_binary(x1, x2)
        logits = logits.squeeze(1)
        loss = criterion_bce(logits, y)
        loss.backward()
        optimizer_bce.step()

        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)
        total_loss += loss.item() * y.size(0)

    el = total_loss / total
    ea = correct / total
    history_binary['loss'].append(el)
    history_binary['acc'].append(ea)
    print(f"  Epoch [{epoch}/{NUM_EPOCHS}] | Loss: {el:.4f} | Accuracy: {ea:.2%}")

print("[✓] Phương pháp 3 huấn luyện xong!\n")


## BƯỚC 8: VẼ BIỂU ĐỒ SO SÁNH QUÁ TRÌNH HUẤN LUYỆN CẢ 3 PHƯƠNG PHÁP


In [ ]:
epochs_x = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Đồ thị hàm mất mát Loss
ax = axes[0, 0]
ax.plot(epochs_x, history_triplet['loss'], 'b-o', label='PP1: Triplet (Random)')
ax.plot(epochs_x, history_hard['loss'], 'r-s', label='PP2: Triplet (Hard Mining)')
ax.plot(epochs_x, history_binary['loss'], 'g-^', label='PP3: Binary Classification')
ax.set_title('So sánh Loss theo Epoch', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Khoảng cách cùng người d(A, P)
ax = axes[0, 1]
ax.plot(epochs_x, history_triplet['d_ap'], 'b-o', label='PP1: d(A,P) Random')
ax.plot(epochs_x, history_hard['d_ap'], 'r-s', label='PP2: d(A,P) Hard Mining')
ax.set_title('Khoảng cách cùng người d(A,P) - Càng nhỏ càng tốt', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('d(A,P)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Khoảng cách khác người d(A, N)
ax = axes[1, 0]
ax.plot(epochs_x, history_triplet['d_an'], 'b-o', label='PP1: d(A,N) Random')
ax.plot(epochs_x, history_hard['d_an'], 'r-s', label='PP2: d(A,N) Hard Mining')
ax.set_title('Khoảng cách khác người d(A,N) - Càng lớn càng tốt', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('d(A,N)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Accuracy của Binary Classification
ax = axes[1, 1]
ax.plot(epochs_x, history_binary['acc'], 'g-^', linewidth=2, label='PP3: Binary Accuracy')
ax.set_title('PP3 Binary Classification: Accuracy theo Epoch', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.4, 1.05)

plt.suptitle('SO SÁNH 3 PHƯƠNG PHÁP FACE RECOGNITION (TỪ SLIDE)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## BƯỚC 9: ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP (ROC CURVE & AUC)
Tạo 500 cặp ảnh test độc lập (250 cặp Cùng người, 250 cặp Khác người) để so sánh định lượng.


In [ ]:
def compute_test_pairs(model, is_binary_model=False, num_pairs=500):
    model.eval()
    scores = []
    true_labels = []

    with torch.no_grad():
        for i in range(num_pairs):
            is_same = (i % 2 == 0)
            if is_same:
                lbl = random.choice(eligible_labels)
                i1, i2 = random.sample(label_to_indices[lbl], 2)
                y = 1
            else:
                l1, l2 = random.sample(all_labels, 2)
                i1 = random.choice(label_to_indices[l1])
                i2 = random.choice(label_to_indices[l2])
                y = 0

            t1 = to_tensor(i1).unsqueeze(0).to(device)
            t2 = to_tensor(i2).unsqueeze(0).to(device)

            if is_binary_model:
                logits, f1, f2 = model(t1, t2)
                score = torch.sigmoid(logits).item()
            else:
                f1 = model(t1)
                f2 = model(t2)
                dist = torch.sum((f1 - f2) ** 2).item()
                # Chuyển khoảng cách sang điểm tương đồng trong [0, 1]
                score = 1.0 / (1.0 + dist)

            scores.append(score)
            true_labels.append(y)

    return np.array(true_labels), np.array(scores)

print("[*] Đang đánh giá 3 phương pháp trên 500 cặp test độc lập...")

y_true_1, scores_1 = compute_test_pairs(model_triplet, is_binary_model=False)
y_true_2, scores_2 = compute_test_pairs(model_hard, is_binary_model=False)
y_true_3, scores_3 = compute_test_pairs(model_binary, is_binary_model=True)

# Tính đường cong ROC và diện tích dưới đường cong (AUC)
fpr1, tpr1, _ = roc_curve(y_true_1, scores_1)
fpr2, tpr2, _ = roc_curve(y_true_2, scores_2)
fpr3, tpr3, _ = roc_curve(y_true_3, scores_3)

auc1 = auc(fpr1, tpr1)
auc2 = auc(fpr2, tpr2)
auc3 = auc(fpr3, tpr3)

plt.figure(figsize=(8, 7))
plt.plot(fpr1, tpr1, 'b-', linewidth=2, label=f'PP1: Triplet Random (AUC = {auc1:.4f})')
plt.plot(fpr2, tpr2, 'r-', linewidth=2, label=f'PP2: Triplet Hard Mining (AUC = {auc2:.4f})')
plt.plot(fpr3, tpr3, 'g-', linewidth=2, label=f'PP3: Binary Classification (AUC = {auc3:.4f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Đường chuẩn ngẫu nhiên (AUC = 0.5)')
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('Đường cong ROC Curve - So sánh 3 Phương pháp', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## BƯỚC 10: TÌM NGƯỠNG TỐI ƯU $\tau$ VÀ SO SÁNH ĐỘ CHÍNH XÁC TỐI ĐA


In [ ]:
def best_accuracy_at_threshold(y_true, scores, num_thresholds=200):
    best_acc = 0
    best_thresh = 0
    for t in np.linspace(0.01, 0.99, num_thresholds):
        preds = (scores >= t).astype(int)
        acc = accuracy_score(y_true, preds)
        if acc > best_acc:
            best_acc = acc
            best_thresh = t
    return best_acc, best_thresh

acc1, th1 = best_accuracy_at_threshold(y_true_1, scores_1)
acc2, th2 = best_accuracy_at_threshold(y_true_2, scores_2)
acc3, th3 = best_accuracy_at_threshold(y_true_3, scores_3)

print("=" * 68)
print(f"{'Phương pháp':<35} {'Accuracy Max':>14} {'Ngưỡng tau':>10} {'AUC':>7}")
print("=" * 68)
print(f"{'PP1: Triplet Loss (Random)':<35} {acc1:>14.2%} {th1:>10.3f} {auc1:>7.4f}")
print(f"{'PP2: Triplet Loss (Hard Mining)':<35} {acc2:>14.2%} {th2:>10.3f} {auc2:>7.4f}")
print(f"{'PP3: Binary Classification':<35} {acc3:>14.2%} {th3:>10.3f} {auc3:>7.4f}")
print("=" * 68)


## BƯỚC 11: TRỰC QUAN HÓA KHÔNG GIAN ĐẶC TRƯNG 128-D BẰNG t-SNE
Quan sát cách mỗi phương pháp gom cụm các khuôn mặt của cùng một người trong không gian đa chiều.


In [ ]:
top5 = sorted(eligible_labels, key=lambda l: len(label_to_indices[l]), reverse=True)[:5]

def extract_embeddings(model, is_binary=False):
    model.eval()
    embs, lbls = [], []
    with torch.no_grad():
        for lbl in top5:
            for idx in label_to_indices[lbl]:
                t = to_tensor(idx).unsqueeze(0).to(device)
                if is_binary:
                    emb = model.embedding_net(t)
                else:
                    emb = model(t)
                embs.append(emb.cpu().numpy().flatten())
                lbls.append(target_names[lbl].split()[-1])
    return np.array(embs), lbls

embs_1, lbls_1 = extract_embeddings(model_triplet)
embs_2, lbls_2 = extract_embeddings(model_hard)
embs_3, lbls_3 = extract_embeddings(model_binary, is_binary=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
titles = [
    f'PP1: Triplet (Random)\nAUC={auc1:.3f}',
    f'PP2: Triplet (Hard Mining)\nAUC={auc2:.3f}',
    f'PP3: Binary Classification\nAUC={auc3:.3f}'
]

for ax, embs, lbls, title in zip(axes, [embs_1, embs_2, embs_3], [lbls_1, lbls_2, lbls_3], titles):
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(10, len(embs)-1))
    emb_2d = tsne.fit_transform(embs)

    unique_names = sorted(set(lbls))
    for name in unique_names:
        mask = [l == name for l in lbls]
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=name, s=55, alpha=0.85)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Trực quan hóa t-SNE: Khả năng gom cụm cùng người trong không gian 128-D', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## BƯỚC 12: BẢNG TỔNG KẾT SO SÁNH VÀ KẾT LUẬN HỌC THUẬT

| Tiêu chí | PP1: Triplet (Random) | PP2: Triplet (Hard Mining) | PP3: Binary Classification |
| :--- | :--- | :--- | :--- |
| **Vị trí trong Slide** | Slide 16-20 | Slide 19, 21 | Slide 24-28 |
| **Cách chọn dữ liệu** | Ngẫu nhiên bộ 3 $(A, P, N)$ | Lọc $P$ xa nhất và $N$ gần nhất | Ghép cặp $(x_1, x_2)$ gán nhãn 0/1 |
| **Hàm mất mát** | $L = \max(d(A,P) - d(A,N) + \alpha, 0)$ | Cùng công thức nhưng trên mẫu khó | $\text{BCE}(\sigma(W \cdot \|f_1 - f_2\| + b), y)$ |
| **Tốc độ hội tụ** | Chậm (nhiều bộ ba quá dễ) | **Nhanh nhất**, học sâu đặc trưng | Trung bình |
| **Khả năng One-Shot** | **Tốt**, khoảng cách L2 trực tiếp | **Tốt nhất**, phân cụm rõ nhất | Cần qua thêm lớp phân loại phụ |
| **Khuyến nghị sử dụng** | Phù hợp hiểu lý thuyết cơ bản | **Khuyên dùng trong thực tế (FaceNet)** | Tốt khi cần đầu ra là xác suất $[0, 1]$ |


## BƯỚC 13: XUẤT TRỌNG SỐ MÔ HÌNH TỐT NHẤT ĐỂ CHẠY WEBCAM OFFLINE


In [ ]:
results = {
    'PP1_Triplet_Random': {'auc': auc1, 'acc': acc1, 'model': model_triplet},
    'PP2_Triplet_Hard':   {'auc': auc2, 'acc': acc2, 'model': model_hard},
    'PP3_Binary':         {'auc': auc3, 'acc': acc3, 'model': model_binary.embedding_net},
}

best_name = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']

torch.save(best_model.state_dict(), 'siamese_triplet.pth')
print(f"[✓] Mô hình tốt nhất là: {best_name} (AUC = {results[best_name]['auc']:.4f})")
print(f"[✓] Đã lưu trọng số vào: siamese_triplet.pth")

# Tự động tải về máy nếu chạy trên Colab
try:
    from google.colab import files
    files.download('siamese_triplet.pth')
    print("[*] Đang tải file trọng số về máy tính...")
except ImportError:
    print("[*] Chạy offline, file đã lưu tại thư mục hiện tại.")
